<a href="https://colab.research.google.com/github/NourBenMadhi/NLP-Project-Movie-Review-Sentiment-Classification/blob/main/projet_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification,get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from torch.optim import AdamW

2026-01-23 19:56:01.273500: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769198161.466908      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769198161.520985      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769198162.013153      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769198162.013194      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769198162.013196      23 computation_placer.cc:177] computation placer alr

In [ ]:
Seed = 42
torch.manual_seed(Seed)
np.random.seed(Seed)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [ ]:
# TODO 1: Load the IMDB dataset CSV file into a pandas DataFrame
# HINT: Use pd.read_csv(...) and the provided Kaggle path
df = pd.read_csv("/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

# TODO 2: Display the dataset shape
# HINT: DataFrame has an attribute that returns (rows, columns)
print("Dataset shape:", df.shape)

# TODO 3: Display the first few rows of the dataset
# HINT: There is a built-in DataFrame method for previewing data
print("\nFirst rows of the dataset:")
print(df.head())

# TODO 4: Print the sentiment distribution
# HINT: Use value_counts() on the sentiment column
print("\nSentiment Distribution:")
print(df['sentiment'].value_counts())

# TODO 5 & 6: Display sample reviews
print("\nSample reviews:")
for i in range(2):
    print(f"\n{i+1}. Sentiment:", df.iloc[i]['sentiment'])
    print("Review:", df.iloc[i]['review'][:200], "...")

Dataset shape: (50000, 2)

First rows of the dataset:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Sentiment Distribution:
sentiment
positive    25000
negative    25000
Name: count, dtype: int64

Sample reviews:

1. Sentiment: positive
Review: One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo ...

2. Sentiment: positive
Review: A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to t

In [ ]:

# TODO 7: Convert sentiment labels into numerical values
# HINT: Use map() with {'positive': 1, 'negative': 0}
df['label'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# TODO 8: Split the dataset into training and temporary sets
# HINT:
# - Use train_test_split
# - test_size should be 0.3
# - Use stratification to preserve class balance
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'].values,
    df['label'].values,
    test_size=0.3,
    random_state=Seed,
    stratify=df['label'].values
)

# TODO 9: Split temporary set into validation and test sets
# HINT: test_size should be 0.5
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=Seed,
    stratify=temp_labels
)
# TODO 10: Print dataset sizes
# HINT: Use len() on each split
print(f"\nTrain size: {len(train_texts)}")
print(f"Validation size: {len(val_texts)}")
print(f"Test size: {len(test_texts)}")


Train size: 35000
Validation size: 7500
Test size: 7500


In [ ]:
# Initialize tokenizer
model_name = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        # TODO 1: Store inputs as class attributes
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        # TODO 2: Return the number of samples in the dataset
        return len(self.texts)

    def __getitem__(self, idx):
        # TODO 3: Retrieve one text sample and its label
        text = str(self.texts[idx])
        label = self.labels[idx]

        # TODO 4: Tokenize the text
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        # TODO 5: Return a dictionary of tensors
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# TODO 6: Create dataset objects for training, validation, and testing
# HINT: Pass texts, labels, tokenizer, and max_length
train_dataset = IMDBDataset(train_texts, train_labels, tokenizer, 256)
val_dataset   = IMDBDataset(val_texts, val_labels, tokenizer, 256)
test_dataset  = IMDBDataset(test_texts, test_labels, tokenizer, 256)

In [ ]:
# TODO 7: Create DataLoaders
# HINT:
# - Training loader should shuffle
# - Validation and test loaders should not
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
# TODO 8: Print the number of batches for each split
# HINT: len(DataLoader) returns number of batches
print(f"\nNumber of training batches: {len(train_loader)}")
print(f"Number of validation batches: {len(val_loader)}")
print(f"Number of testing batches: {len(test_loader)}")


Number of training batches: 2188
Number of validation batches: 469
Number of testing batches: 469


In [ ]:
# TODO 1: Load a pre-trained DistilBERT model for sequence classification
# HINT: Use from_pretrained(...) and specify the number of labels
model = DistilBertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# TODO 2: Move the model to the correct device (CPU or GPU)
# HINT: Use .to(device)
model = model.to(device)

# Training hyperparameters
epochs = 3
lr = 0.01  # Correct learning rate for DistilBERT fine-tuning

# TODO 3: Initialize the optimizer
# HINT: AdamW is commonly used for Transformer fine-tuning
optimizer = AdamW(model.parameters(), lr=lr)

# TODO 4: Compute the total number of training steps
# HINT: Number of batches × number of epochs
total_steps = len(train_loader) * epochs

# TODO 5: Initialize the learning rate scheduler
# HINT: Use a linear schedule with warmup
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# TODO 6: Print model statistics
# HINT: p.numel() gives the number of parameters in a tensor
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training for {epochs} epochs with learning rate {lr}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Model parameters: 66,955,010
Training for 3 epochs with learning rate 0.01


In [ ]:
def train(model, data_loader, optimizer, scheduler, device):
    # TODO 7: Set the model to training mode
    model.train()

    losses = []
    correct_predictions = 0

    progress_bar = tqdm(data_loader, desc="Training")

    for batch in progress_bar:
        # TODO 8: Move batch data to the correct device
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Clear previously calculated gradients
        optimizer.zero_grad()

        # TODO 9: Forward pass
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        # TODO 10: Extract loss and logits
        loss = outputs.loss
        logits = outputs.logits

        # TODO 11: Compute predictions
        _, preds = torch.max(logits, dim=1)

        # TODO 12: Count correct predictions
        correct_predictions += torch.sum(preds == labels)

        losses.append(loss.item())

        # TODO 13: Backpropagation
        loss.backward()

        # TODO 14: Gradient clipping
        # HINT: Prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        # TODO 15: Optimizer and scheduler steps
        optimizer.step()
        scheduler.step()

        progress_bar.set_postfix({"loss": loss.item()})

    # TODO 16: Return accuracy and mean loss
    return (
        correct_predictions.double() / len(data_loader.dataset),
        np.mean(losses)
    )


In [ ]:
def eval(model, data_loader, device):

    # Disable training-specific layers such as dropout
    model.eval()

    losses = []
    correct_predictions = 0

    # Disable gradient tracking entirely
    with torch.no_grad():

        for batch in tqdm(data_loader, desc="Evaluating"):

            # Same extraction logic as training
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Forward pass (loss can still be computed)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            # Extract loss and logits
            loss = outputs.loss
            logits = outputs.logits

            # Compute predictions
            _, preds = torch.max(logits, dim=1)

            # Accumulate correct predictions
            correct_predictions += torch.sum(preds == labels)

            # Store loss
            losses.append(loss.item())

    # Accuracy over full dataset and mean evaluation loss
    return (
        correct_predictions.double() / len(data_loader.dataset),
        np.mean(losses)
    )

In [ ]:
history = {
    "train_acc": [],
    "train_loss": [],
    "val_acc": [],
    "val_loss": []
}

best_val_acc = 0  # Initialize with the worst possible accuracy

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1} / {epochs}")

    # Train the model for one epoch
    train_acc, train_loss = train(model, train_loader, optimizer, scheduler, device)
    print(f"Train loss: {train_loss:.2f} | Train Accuracy: {train_acc:.2f}")

    # Evaluate the model on validation data
    val_acc, val_loss = eval(model, val_loader, device)
    print(f"Val loss: {val_loss:.2f} | Val Accuracy: {val_acc:.2f}")

    # Store metrics in history
    history["train_acc"].append(train_acc.item())
    history["train_loss"].append(train_loss)
    history["val_acc"].append(val_acc.item())
    history["val_loss"].append(val_loss)

    # Save the model if validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "model.pt")
        print(f"Model Saved (Val Accuracy: {val_acc:.2f})")




Epoch 1 / 3


Training: 100%|██████████| 2188/2188 [09:24<00:00,  3.88it/s, loss=0.672]


Train loss: 0.70 | Train Accuracy: 0.50


Evaluating: 100%|██████████| 469/469 [00:53<00:00,  8.73it/s]


Val loss: 0.69 | Val Accuracy: 0.50
Model Saved (Val Accuracy: 0.50)

Epoch 2 / 3


...

Val loss: 0.69 | Val Accuracy: 0.50

Epoch 3 / 3


Training: 100%|██████████| 2188/2188 [09:22<00:00,  3.89it/s, loss=0.694]


Train loss: 0.69 | Train Accuracy: 0.50


Evaluating: 100%|██████████| 469/469 [00:53<00:00,  8.72it/s]

Val loss: 0.69 | Val Accuracy: 0.50


In [ ]:
# Accuracy plot
ax1.plot(history["train_acc"], label="Train Accuracy", marker="o")
# HINT: Use training accuracy history

ax1.plot(history["val_acc"], label="Val Accuracy", marker="o")
# HINT: Use validation accuracy history

ax1.set_xlabel("epoch")
# HINT: X-axis represents training progression

ax1.set_ylabel("Accuracy")
# HINT: Metric being measured

ax1.set_title("Accuracy")
# HINT: Describe what the plot shows

ax1.legend()
ax1.grid(True)

# Loss plot
ax2.plot(history["train_loss"], label="Train Loss", marker="o")
# HINT: Use training loss history

ax2.plot(history["val_loss"], label="Val Loss", marker="o")
# HINT: Use validation loss history

ax2.set_xlabel("epoch")
# HINT: Same x-axis meaning as above

ax2.set_ylabel("loss")
# HINT: Loss magnitude

ax2.set_title("Loss")
# HINT: Describe loss behavior over epochs

ax2.legend()
ax2.grid(True)

plt.tight_layout()
# HINT: Prevent overlapping plot elements

plt.show()
# HINT: Render the figures

NameError: name 'ax1' is not defined

In [ ]:
# Load the best-performing model weights
model.load_state_dict(torch.load("model.pt"))
# HINT: Restore the parameters saved during training

# Evaluate on the test set
test_acc, test_loss = eval(model, test_loader, device)
# HINT: Use the evaluation function with the test data loader

print(f"Test Accuracy:  {test_acc.item():.2f}")
# HINT: Accuracy should be formatted as a floating-point value

print(f"Test Loss: {test_loss:.2f}")
# HINT: Loss should be displayed as a scalar value